# Experimentos - MLP do Zero

Este notebook registra os experimentos comparativos da implementação do MLP. A primeira parte usa um dataset sintético simples para validar o fluxo. A segunda parte prepara o treinamento real no MNIST.

## Objetivo

Comparar arquiteturas diferentes usando métricas de loss e acurácia. Primeiro faço isso em um dataset controlado. Depois aplico a mesma lógica ao MNIST, que é o dataset obrigatório da atividade.

In [ ]:
from pathlib import Path
import csv
import sys

import numpy as np

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from mlp.losses import cross_entropy_loss, one_hot_encode
from mlp.network import MLP

## Dataset Sintético

O dataset abaixo tem duas classes separáveis. Ele é pequeno de propósito: se o MLP não aprender este problema, é melhor investigar a implementação antes de usar MNIST.

In [ ]:
X_train = np.array([
    [-1.0, -1.0],
    [-1.0, -0.5],
    [-0.5, -1.0],
    [-0.8, -0.7],
    [1.0, 1.0],
    [1.0, 0.5],
    [0.5, 1.0],
    [0.8, 0.7],
])
y_train = np.array([0, 0, 0, 0, 1, 1, 1, 1])

X_val = np.array([
    [-0.9, -0.8],
    [-0.6, -0.9],
    [0.9, 0.8],
    [0.6, 0.9],
])
y_val = np.array([0, 0, 1, 1])

X_train.shape, y_train.shape, X_val.shape, y_val.shape

## Configurações Comparadas no Dataset Sintético

- Configuração A: rede menor, learning rate maior.
- Configuração B: rede maior, learning rate menor.

A comparação ainda não busca a melhor arquitetura final. O objetivo é confirmar que o fluxo de experimentos funciona.

In [ ]:
synthetic_configurations = [
    {
        "nome": "Configuração A",
        "arquitetura": [2, 8, 4, 2],
        "learning_rate": 0.1,
        "batch_size": 4,
        "epochs": 40,
        "seed": 42,
    },
    {
        "nome": "Configuração B",
        "arquitetura": [2, 16, 8, 2],
        "learning_rate": 0.05,
        "batch_size": 4,
        "epochs": 40,
        "seed": 42,
    },
]

In [ ]:
def run_synthetic_experiment(config):
    model = MLP(
        layer_sizes=config["arquitetura"],
        learning_rate=config["learning_rate"],
        seed=config["seed"],
    )
    history = model.fit(
        X_train,
        y_train,
        epochs=config["epochs"],
        batch_size=config["batch_size"],
        shuffle=True,
        X_val=X_val,
        y_val=y_val,
    )

    return {
        "nome": config["nome"],
        "arquitetura": " -> ".join(str(size) for size in config["arquitetura"]),
        "learning_rate": config["learning_rate"],
        "batch_size": config["batch_size"],
        "epochs": config["epochs"],
        "loss_final": history["loss"][-1],
        "accuracy_final": history["accuracy"][-1],
        "val_loss_final": history["val_loss"][-1],
        "val_accuracy_final": history["val_accuracy"][-1],
    }


synthetic_results = [run_synthetic_experiment(config) for config in synthetic_configurations]
synthetic_results

In [ ]:
header = (
    "Configuração | Arquitetura | LR | Batch | Épocas | "
    "Loss Treino | Acc Treino | Loss Val | Acc Val"
)
separator = "--- | --- | --- | --- | --- | --- | --- | --- | ---"
rows = []

for result in synthetic_results:
    rows.append(
        "{nome} | {arquitetura} | {learning_rate} | {batch_size} | {epochs} | "
        "{loss_final:.6f} | {accuracy_final:.6f} | "
        "{val_loss_final:.6f} | {val_accuracy_final:.6f}".format(**result)
    )

print(header)
print(separator)
for row in rows:
    print(row)

## Treinamento no MNIST

Nesta seção uso `tensorflow.keras.datasets.mnist` apenas para carregar o dataset. Todo o treinamento continua sendo feito pela classe `MLP`, implementada manualmente com NumPy.

### Carregamento e Pré-processamento

Cada imagem do MNIST tem formato 28x28. Para entrar no MLP, cada imagem é achatada para um vetor de 784 valores. Também normalizo os pixels de 0-255 para 0-1.

In [ ]:
try:
    from tensorflow.keras.datasets import mnist
except ImportError as exc:
    raise ImportError(
        "TensorFlow/Keras não está instalado. Rode `pip install -r requirements.txt` "
        "antes de executar a seção MNIST."
    ) from exc

(X_train_full, y_train_full), (X_test, y_test) = mnist.load_data()

X_train_full = X_train_full.reshape(X_train_full.shape[0], -1).astype("float32") / 255.0
X_test = X_test.reshape(X_test.shape[0], -1).astype("float32") / 255.0

validation_size = 10000
X_val_mnist = X_train_full[-validation_size:]
y_val_mnist = y_train_full[-validation_size:]
X_train_mnist = X_train_full[:-validation_size]
y_train_mnist = y_train_full[:-validation_size]

X_train_mnist.shape, X_val_mnist.shape, X_test.shape

### Configurações para MNIST

As duas configurações abaixo têm pelo menos duas camadas ocultas, como exigido na atividade. A Configuração A é menor. A Configuração B tem mais neurônios e learning rate menor.

In [ ]:
mnist_configurations = [
    {
        "nome": "MNIST A",
        "arquitetura": [784, 128, 64, 10],
        "learning_rate": 0.1,
        "batch_size": 128,
        "epochs": 12,
        "seed": 42,
    },
    {
        "nome": "MNIST B",
        "arquitetura": [784, 256, 128, 10],
        "learning_rate": 0.05,
        "batch_size": 128,
        "epochs": 12,
        "seed": 42,
    },
]

In [ ]:
def run_mnist_experiment(config):
    model = MLP(
        layer_sizes=config["arquitetura"],
        learning_rate=config["learning_rate"],
        seed=config["seed"],
    )
    history = model.fit(
        X_train_mnist,
        y_train_mnist,
        epochs=config["epochs"],
        batch_size=config["batch_size"],
        shuffle=True,
        X_val=X_val_mnist,
        y_val=y_val_mnist,
    )

    y_test_one_hot = one_hot_encode(y_test, 10)
    test_probabilities = model.predict_proba(X_test)
    test_loss = cross_entropy_loss(test_probabilities, y_test_one_hot)
    test_accuracy = model.accuracy(X_test, y_test)

    return {
        "nome": config["nome"],
        "arquitetura": " -> ".join(str(size) for size in config["arquitetura"]),
        "learning_rate": config["learning_rate"],
        "batch_size": config["batch_size"],
        "epochs": config["epochs"],
        "history": history,
        "loss_final": history["loss"][-1],
        "accuracy_final": history["accuracy"][-1],
        "val_loss_final": history["val_loss"][-1],
        "val_accuracy_final": history["val_accuracy"][-1],
        "test_loss": test_loss,
        "test_accuracy": test_accuracy,
    }


mnist_results = [run_mnist_experiment(config) for config in mnist_configurations]
mnist_results

### Tabela Comparativa do MNIST

A meta da atividade é alcançar pelo menos 92% de acurácia no conjunto de teste.

In [ ]:
mnist_table_header = (
    "Configuração | Arquitetura | LR | Batch | Épocas | "
    "Loss Val | Acc Val | Loss Teste | Acc Teste"
)
mnist_table_separator = "--- | --- | --- | --- | --- | --- | --- | --- | ---"

print(mnist_table_header)
print(mnist_table_separator)
for result in mnist_results:
    print(
        "{nome} | {arquitetura} | {learning_rate} | {batch_size} | {epochs} | "
        "{val_loss_final:.6f} | {val_accuracy_final:.6f} | "
        "{test_loss:.6f} | {test_accuracy:.6f}".format(**result)
    )

### Salvando Resultados do MNIST

Esta célula salva uma tabela CSV e os gráficos de loss e acurácia em `results/`.

In [ ]:
import matplotlib.pyplot as plt

results_dir = project_root / "results"
results_dir.mkdir(exist_ok=True)

csv_path = results_dir / "mnist_experimentos.csv"
with csv_path.open("w", newline="", encoding="utf-8") as file:
    fieldnames = [
        "configuracao",
        "arquitetura",
        "learning_rate",
        "batch_size",
        "epochs",
        "val_loss",
        "val_accuracy",
        "test_loss",
        "test_accuracy",
    ]
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    for result in mnist_results:
        writer.writerow({
            "configuracao": result["nome"],
            "arquitetura": result["arquitetura"],
            "learning_rate": result["learning_rate"],
            "batch_size": result["batch_size"],
            "epochs": result["epochs"],
            "val_loss": result["val_loss_final"],
            "val_accuracy": result["val_accuracy_final"],
            "test_loss": result["test_loss"],
            "test_accuracy": result["test_accuracy"],
        })

plt.figure(figsize=(8, 5))
for result in mnist_results:
    plt.plot(result["history"]["loss"], label=f"{result['nome']} - treino")
    plt.plot(result["history"]["val_loss"], linestyle="--", label=f"{result['nome']} - validação")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.title("Curva de Loss - MNIST")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(results_dir / "mnist_loss.png", dpi=150)
plt.show()

plt.figure(figsize=(8, 5))
for result in mnist_results:
    plt.plot(result["history"]["accuracy"], label=f"{result['nome']} - treino")
    plt.plot(result["history"]["val_accuracy"], linestyle="--", label=f"{result['nome']} - validação")
plt.xlabel("Época")
plt.ylabel("Acurácia")
plt.title("Curva de Acurácia - MNIST")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(results_dir / "mnist_accuracy.png", dpi=150)
plt.show()

csv_path, results_dir / "mnist_loss.png", results_dir / "mnist_accuracy.png"

## Observação Final

Depois de executar a seção MNIST, os valores obtidos devem ser copiados para o README final. Se nenhuma configuração alcançar 92% de acurácia no teste, o próximo passo é ajustar learning rate, batch size, quantidade de épocas ou número de neurônios.